In [0]:
import time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, count

# 1. INITIALIZE GOLD SESSION
print("🚀 Gold Engine Online. Loading Silver Data...")

try:
    # Ensure Spark session is active (for Spark Connect on serverless)
    try:
        # Test if spark session is active
        _ = spark.version
    except:
        # If not active, get the active session
        spark = SparkSession.getActiveSession()
        if spark is None:
            raise Exception("No active Spark session available. Please restart the notebook.")
    
    # 2. LOAD REFINED DATA FROM UNITY CATALOG
    df_silver = spark.table("`prism-sentinel-stream`.prism_silver.transactions_refined")
    
    # 3. EXTRACT RISK POPULATIONS (Optimization)
    # The table uses risk_level (CRITICAL, HIGH, MEDIUM, LOW) instead of numeric scores
    # We'll map these to probabilities for the simulation
    print("📊 Aggregating Risk Profiles...")
    risk_counts = df_silver.groupBy("risk_level").count().collect()
    
    # Convert to a dictionary for easy access: {'CRITICAL': 15021, 'HIGH': 199753, ...}
    risk_profile = {row['risk_level']: row['count'] for row in risk_counts}
    
    count_critical = risk_profile.get('CRITICAL', 0)
    count_high = risk_profile.get('HIGH', 0)
    count_medium = risk_profile.get('MEDIUM', 0)
    
    print(f"   - Critical Risk Entities: {count_critical:,}")
    print(f"   - High Risk Entities: {count_high:,}")
    print(f"   - Medium Risk Entities: {count_medium:,}")

    # 4. MONTE CARLO SIMULATION (Binomial Method)
    # Scenario:
    # - CRITICAL Risk = Sanctions Violation = $50,000 Fine (95% probability)
    # - HIGH Risk = Sanctions Evasion = $50,000 Fine (70% probability)
    # - MEDIUM Risk = Smurfing Suspicion = $10,000 Fine (5% probability if proven)
    
    NUM_SIMULATIONS = 5000
    FINE_SANCTIONS = 50000
    FINE_SMURFING = 10000
    
    print(f"\n🎲 Running {NUM_SIMULATIONS} Monte Carlo scenarios...")
    start_sim = time.time()
    
    results = []
    
    for _ in range(NUM_SIMULATIONS):
        # We use a Binomial distribution: "Flip a coin N times with probability P"
        # This is mathematically identical to iterating rows but instantaneous.
        
        # Simulating confirmed critical sanctions hits (95% probability)
        confirmed_critical = np.random.binomial(n=count_critical, p=0.95)
        
        # Simulating confirmed high risk sanctions (70% probability)
        confirmed_high = np.random.binomial(n=count_high, p=0.70)
        
        # Simulating confirmed smurfing cases (5% probability of conversion to fine)
        confirmed_smurfing = np.random.binomial(n=count_medium, p=0.05) 
        
        total_fine = ((confirmed_critical + confirmed_high) * FINE_SANCTIONS) + (confirmed_smurfing * FINE_SMURFING)
        results.append(total_fine)
        
    print(f"✅ Simulation Complete in {round(time.time() - start_sim, 2)}s")

    # 5. CALCULATE VALUE AT RISK (VaR)
    results = np.array(results)
    mean_exposure = np.mean(results)
    var_95 = np.percentile(results, 95) # 95% Confidence Interval
    var_99 = np.percentile(results, 99) # 99% Confidence Interval (Stress Test)

    # 6. EXECUTIVE REGULATORY REPORT
    print("\n" + "="*60)
    print("🛡️ PRISM-RISK SENTINEL: REGULATORY CAPITAL REPORT")
    print("="*60)
    print(f"✅ Total Transactions Scanned: {df_silver.count():,}")
    print(f"🚨 Critical Sanctions Hits: {count_critical:,}")
    print(f"⚠️ High Risk (Evasion Patterns): {count_high:,}")
    print(f"⚠️ Medium Risk (Smurfing Patterns): {count_medium:,}")
    print("-" * 60)
    print("💰 PROJECTED REGULATORY FINES (Probabilistic Model):")
    print(f"   • Expected Mean Loss:      ${mean_exposure:,.2f}")
    print(f"   • VaR (95% Confidence):    ${var_95:,.2f}")
    print(f"   • VaR (99% Stress Test):   ${var_99:,.2f}")
    print("-" * 60)
    print("📝 STRATEGIC RECOMMENDATION:")
    
    if var_95 > 100000000: # $100M Threshold
        print("🔴 CRITICAL: Insolvency Risk. Immediate capital injection required.")
    elif var_95 > 10000000: # $10M Threshold
        print("🟠 WARNING: Reserve Capital insufficient. Increase provisions by 15%.")
    else:
        print("🟢 STABLE: Current capital reserves cover 99% of risk scenarios.")
    print("="*60)

    # 7. VISUALIZATION
    plt.figure(figsize=(12, 6))
    plt.hist(results, bins=50, color='#2c3e50', alpha=0.7, edgecolor='black')
    plt.axvline(mean_exposure, color='gold', linestyle='--', linewidth=2, label=f'Expected: ${mean_exposure/1e6:.1f}M')
    plt.axvline(var_95, color='red', linestyle='-', linewidth=2, label=f'VaR 95%: ${var_95/1e6:.1f}M')
    
    plt.title('Monte Carlo Simulation: Regulatory Fine Exposure Distribution', fontsize=14)
    plt.xlabel('Total Estimated Fines ($)', fontsize=12)
    plt.ylabel('Frequency (Likelihood)', fontsize=12)
    plt.legend()
    plt.grid(True, alpha=0.3)
    plt.show()

except Exception as e:
    print(f"❌ Simulation Failed: {e}")
    import traceback
    traceback.print_exc()

In [0]:
from pyspark.sql.functions import col, lit, expr, when, sum

# 1. We take our cleaned Silver data
# 2. We 'cross join' with a set of 10,000 randomized risk scenarios
# 3. We count how many times the risk score exceeds our 'Alert' threshold

# Generate 10,000 random scenarios
scenarios_df = spark.range(10000).withColumn("scenario_id", col("id")).drop("id")

# Cross join transactions with scenarios to simulate volatility per scenario
cross_df = df_silver.select("transaction_id", "amount") \
    .crossJoin(scenarios_df) \
    .withColumn("simulated_volatility", expr("randn() * 0.1")) \
    .withColumn("simulated_amount", col("amount") * (1 + col("simulated_volatility"))) \
    .withColumn("is_hit", when(col("simulated_amount") > 10000, 1).otherwise(0))

# Grouping to get the final simulation_hits count per transaction
simulation_results = cross_df.groupBy("transaction_id") \
    .agg(sum("is_hit").alias("simulation_hits"))

print(simulation_results.head(5))

In [0]:
# Save simulation results to gold table
from pyspark.sql.functions import col, current_timestamp, lit
from pyspark.sql.types import DoubleType

# Convert numpy array results to a Spark DataFrame
# Create a DataFrame with potential_exposure column
results_df = spark.createDataFrame(
    [(float(val),) for val in results],
    ["potential_exposure"]
)
print(results_df.head())
# Save to the risk_simulation_results table
(results_df.write
    .format("delta")
    .mode("overwrite")
    .option("mergeSchema", "true")
    .saveAsTable("`prism-sentinel-stream`.prism_gold.risk_simulation_results"))

print(f"✅ Saved {len(results)} simulation results to prism_gold.risk_simulation_results")

In [0]:
# Assuming 'simulation_results' is the output of your Monte Carlo simulation
# It should contain the transaction_id and the count of risk 'hits'

from pyspark.sql.functions import col, current_timestamp

gold_df = simulation_results.withColumn("simulation_hits", col("simulation_hits").cast("int")) \
                    .withColumn("fraud_probability", col("simulation_hits") / 10000.0) \
                    .withColumn("processed_timestamp", current_timestamp())

(gold_df.write
    .format("delta")
    .mode("overwrite") # Or append, depending on your reporting window
    .option("mergeSchema", "true")
    .saveAsTable("`prism-sentinel-stream`.prism_gold.risk_gold_monte_carlo"))

    print(f"✅ Saved {gold_df.count()} records to prism_gold.risk_gold_monte_carlo")